In [ ]:
import os
import numpy as np
import chromadb
import uuid
import json as _json
from pathlib import Path
from typing import List, Dict, Any
from pypdf import PdfReader

from langchain_google_genai import ChatGoogleGenerativeAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_docling.loader import DoclingLoader, ExportType
from langchain_text_splitters import RecursiveCharacterTextSplitter

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Load environment variables from .env
def _load_env_file(path='../.env'):
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

_load_env_file()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set")


def build_pdf_converter() -> DocumentConverter:
    pdf_options = PdfPipelineOptions(
        force_backend_text=True,
        do_ocr=False,
        do_table_structure=True,
        do_code_enrichment=False,
        do_formula_enrichment=False,
        do_picture_classification=False,
        do_picture_description=True,
        do_chart_extraction=False,
        generate_page_images=False,
        generate_picture_images=False,
        generate_table_images=False,
        generate_parsed_pages=False,
        document_timeout=120,
        ocr_batch_size=1,
        layout_batch_size=1,
        table_batch_size=1,
        batch_polling_interval_seconds=0.1,
        queue_max_size=1,
        images_scale=1.0,
    )
    return DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)},
    )


def load_pdf_documents(pdf_file: Path, converter: DocumentConverter, page_chunk_size: int = 1):
    all_documents = []
    page_count = len(PdfReader(str(pdf_file)).pages)

    for page_start in range(1, page_count + 1, page_chunk_size):
        page_end = min(page_start + page_chunk_size - 1, page_count)
        loader = DoclingLoader(
            str(pdf_file),
            converter=converter,
            export_type=ExportType.DOC_CHUNKS,
            convert_kwargs={"page_range": (page_start, page_end)},
        )
        documents = loader.load()

        for doc in documents:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'
            doc.metadata['source_page_range'] = f"{page_start}-{page_end}"

        all_documents.extend(documents)

    return all_documents


light_pdf_converter = build_pdf_converter()
default_docling_converter = DocumentConverter()


In [2]:
### Read all documents (PDF, PPT, Word, Text) using DoclingLoader
def process_all_documents(directory):
    """Process all supported document files in a directory using Docling"""
    all_documents = []
    doc_dir = Path(directory)
    
    supported_extensions = ['*.pdf', '*.ppt', '*.pptx', '*.doc', '*.docx', '*.txt']
    
    all_files = []
    for ext in supported_extensions:
        all_files.extend(doc_dir.glob(f"**/{ext}"))
    
    print(f"Found {len(all_files)} files to process")
    
    for file_path in all_files:
        file_ext = file_path.suffix.lower()
        print(f"\nProcessing: {file_path.name} ({file_ext})")
        try:
            if file_ext == '.pdf':
                documents = load_pdf_documents(file_path, light_pdf_converter, page_chunk_size=1)
            else:
                converter = default_docling_converter
                loader = DoclingLoader(
                    str(file_path),
                    converter=converter,
                    export_type=ExportType.DOC_CHUNKS,
                )
                documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = file_path.name
                doc.metadata['file_type'] = file_ext[1:]
                
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} chunks")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


all_documents = process_all_documents("../data")


Found 3 files to process

Processing: base research paper.pdf (.pdf)


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


  ✓ Loaded 102 chunks

Processing: main_paper.pdf (.pdf)
  ✓ Loaded 35 chunks

Processing: AI.txt (.txt)
  ✓ Loaded 2 chunks

Total documents loaded: 139


In [3]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\ANISH\AppData\Local\Temp\ipykernel_30556\961631792.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [4]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata - flatten useful fields
            metadata = {}
            for key, value in doc.metadata.items():
                if isinstance(value, (str, int, float, bool, list)) or value is None:
                    metadata[key] = value
                elif key == 'origin' and isinstance(value, dict):
                    # Extract filename from origin
                    metadata['filename'] = value.get('filename', '')
                elif key == 'headings' and isinstance(value, list):
                    # Store headings as comma-separated string for filtering
                    metadata['headings'] = ', '.join(value) if value else ''
                elif key == 'dl_meta' and isinstance(value, dict):
                    # Extract page numbers from doc_items if available
                    doc_items = value.get('doc_items', [])
                    if doc_items and len(doc_items) > 0:
                        first_item = doc_items[0]
                        prov = first_item.get('prov', [])
                        if prov and len(prov) > 0:
                            metadata['page_no'] = prov[0].get('page_no', 0)
                    # Store full dl_meta as JSON for retrieval
                    import json
                    metadata['dl_meta'] = json.dumps(value)
                else:
                    # Store other complex metadata as JSON for retrieval
                    import json
                    metadata[key] = json.dumps(value)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 262


In [5]:
# Create text chunks from the loaded documents
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_documents)
print(f"Created {len(chunks)} chunks from {len(all_documents)} documents")


Created 169 chunks from 139 documents


In [6]:
class RAGRetriever:
    """Enhanced retriever with hybrid search and context optimization"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0, 
                 filter_metadata: Dict = None) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query with enhanced ranking
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            filter_metadata: Optional metadata filters (e.g., {'file_type': 'pdf'})
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        if filter_metadata:
            print(f"Filters: {filter_metadata}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store with optional filters
        try:
            query_params = {
                "query_embeddings": [query_embedding.tolist()],
                "n_results": top_k * 2  # Retrieve more for re-ranking
            }
            
            if filter_metadata:
                query_params["where"] = filter_metadata
            
            results = self.vector_store.collection.query(**query_params)
            
            # Process and re-rank results
            retrieved_docs = self._process_and_rank_results(results, query, top_k, score_threshold)
            
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering and re-ranking)")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    
    def _process_and_rank_results(self, results: Dict, query: str, top_k: int, 
                                  score_threshold: float) -> List[Dict[str, Any]]:
        """Process, deduplicate, and re-rank results"""
        retrieved_docs = []
        
        if not results['documents'] or not results['documents'][0]:
            print("No documents found")
            return []
        
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]
        
        # Track seen content to avoid duplicates
        seen_content = set()
        
        for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
            # Convert distance to similarity score
            similarity_score = 1 - distance
            
            if similarity_score < score_threshold:
                continue
            
            # Deduplicate by content hash
            content_hash = hash(document)
            if content_hash in seen_content:
                continue
            seen_content.add(content_hash)
            
            # Calculate enhanced score with metadata weighting
            enhanced_score = self._calculate_enhanced_score(document, query, similarity_score, metadata)
            
            retrieved_docs.append({
                'id': doc_id,
                'content': document,
                'metadata': metadata,
                'similarity_score': similarity_score,
                'enhanced_score': enhanced_score,
                'distance': distance,
                'rank': i + 1
            })
        
        # Sort by enhanced score and return top_k
        retrieved_docs.sort(key=lambda x: x['enhanced_score'], reverse=True)
        return retrieved_docs[:top_k]
    
    def _calculate_enhanced_score(self, document: str, query: str, 
                                 similarity_score: float, metadata: Dict) -> float:
        """Calculate enhanced relevance score using multiple signals"""
        score = similarity_score
        
        # Boost score if query terms appear in headings
        headings = metadata.get('headings', '')
        if headings:
            query_terms = set(query.lower().split())
            heading_terms = set(headings.lower().split())
            overlap = len(query_terms & heading_terms)
            if overlap > 0:
                score += 0.1 * overlap  # Boost for heading matches
        
        # Boost score for longer, more comprehensive content
        content_length = len(document)
        if content_length > 500:
            score += 0.05
        elif content_length > 1000:
            score += 0.1
        
        # Boost score for recent documents (if timestamp available)
        # This can be extended based on your metadata
        
        return min(score, 1.0)  # Cap at 1.0
    
    def format_context_for_llm(self, retrieved_docs: List[Dict[str, Any]], 
                               max_context_length: int = 4000) -> str:
        """
        Format retrieved documents into optimized context for LLM
        
        Args:
            retrieved_docs: List of retrieved documents
            max_context_length: Maximum context length in characters
            
        Returns:
            Formatted context string
        """
        if not retrieved_docs:
            return "No relevant context found."
        
        context_parts = []
        current_length = 0
        
        for doc in retrieved_docs:
            # Extract metadata for citation
            filename = doc['metadata'].get('filename', doc['metadata'].get('source_file', 'Unknown'))
            page_no = doc['metadata'].get('page_no', 'N/A')
            headings = doc['metadata'].get('headings', '')
            
            # Format document with metadata
            doc_header = f"[Source: {filename}"
            if page_no != 'N/A':
                doc_header += f", Page {page_no}"
            if headings:
                doc_header += f", Section: {headings}"
            doc_header += "]"
            
            doc_content = f"{doc_header}\n{doc['content']}"
            
            # Check if adding this document would exceed limit
            if current_length + len(doc_content) > max_context_length:
                break
            
            context_parts.append(doc_content)
            current_length += len(doc_content)
        
        # Join with clear separators
        context = "\n\n---\n\n".join(context_parts)
        
        # Add summary statistics
        context = f"Context from {len(context_parts)} document(s):\n\n{context}"
        
        return context

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [7]:
def ask_gemini(query, retriever, llm, top_k: int = 3, max_context_length: int = 4000, 
               filter_metadata: Dict = None):
    """Ask Gemini with enhanced retrieval and context formatting."""
    relevant_docs = retriever.retrieve(query, top_k=top_k, filter_metadata=filter_metadata)
    
    if not relevant_docs:
        return "I don't have enough relevant information to answer this question."
    
    context_text = retriever.format_context_for_llm(relevant_docs, max_context_length)
    
    prompt = f"""You are a helpful assistant that answers questions based on the provided context.
The context contains information from various documents with source citations.

Context:
{context_text}

Question: {query}

Instructions:
- Answer the question using only the provided context
- If the answer is not in the context, say you don't have enough information
- Include source citations in your answer when relevant
- Be specific and accurate
- If multiple documents provide information, synthesize them coherently

Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

Integrating LLM

In [8]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set")

# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3, api_key=GEMINI_API_KEY)


In [9]:
question = "What is Carbon-aware Resource Scheduling?"
answer = ask_gemini(question, rag_retriever, llm, top_k=5, max_context_length=5000)
print(answer)

Retrieving documents for query: 'What is Carbon-aware Resource Scheduling?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering and re-ranking)
Carbon-aware resource scheduling involves shifting flexible workloads in time or space towards lower-carbon electricity, by exploiting temporal or spatial variations in grid carbon intensity [main_paper.pdf]. The objective is to minimize carbon emissions, often alongside other metrics like average or tail Job Completion Time (JCT), P99 latency, and starvation [main_paper.pdf].

This type of scheduling is particularly suited for delay-tolerant batch jobs, such as model training, log aggregation, backups, scientific simulations, and CI/CD workloads, rather than high-priority interactive services with strict latency requirements [main_paper.pdf].

Prior approaches have included geographic load balancing, temporal shifting, and carbon-intensity-aware resource management [main_paper.pdf]. A key challenge is to reduce carbon emissions without violating service-level objectives or overlooking reso

In [18]:
import os
import numpy as np

# ─── 1. TruLens Imports (Verified for TruLens 2.8.1) ───────────────────────────
from trulens.core import TruSession, Feedback, Selector
from trulens.apps.custom import TruCustomApp
from trulens.apps.app import instrument
from trulens.providers.google import Google

# Initialize TruLens Session and clear any old databases
tru = TruSession()
tru.reset_database()

# ─── 2. Set Up the Evaluator (Gemini) ──────────────────────────────────────────
google_provider = Google(model_engine="gemini-2.5-flash")

# ─── 3. Define the Feedback Functions ─────────────────────────────────────────

# Metric 1: Answer Relevance
f_answer_relevance = (
    Feedback(google_provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on({
        "prompt": Selector.select_record_input(),
        "response": Selector.select_record_output()
    })
)

# Metric 2: Context Relevance
f_context_relevance = (
    Feedback(google_provider.context_relevance_with_cot_reasons, name="Context Relevance")
    .on({
        "question": Selector.select_record_input(),
        "context": Selector.select_context(collect_list=True)
    })
)

# Metric 3: Groundedness
f_groundedness = (
    Feedback(google_provider.groundedness_measure_with_cot_reasons, name="Groundedness")
    .on({
        "source": Selector.select_context(collect_list=True),
        "statement": Selector.select_record_output()
    })
)


# ─── 4. Wrap Your Custom RAG Pipeline ─────────────────────────────────────────
class TruLensRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    @instrument
    def retrieve(self, query: str) -> list:
        """Fetch documents. The @instrument decorator tells TruLens to track this."""
        docs = self.retriever.retrieve(query, top_k=3)
        return docs

    @instrument
    def generate(self, query: str, docs: list) -> str:
        """Generate an answer."""
        context_text = self.retriever.format_context_for_llm(docs, 4000)
        
        prompt = f"""You are a helpful assistant that answers questions based on the provided context.

Context:
{context_text}

Question: {query}

Instructions:
- Answer the question using only the provided context
- If the answer is not in the context, say you don't have enough information
- Include source citations in your answer when relevant
- Be specific and accurate

Answer:"""
        
        response = self.llm.invoke(prompt)
        return response.content

    @instrument
    def query(self, query: str) -> str:
        """The main entry point for our app."""
        docs = self.retrieve(query)
        if not docs:
            return "I don't have enough relevant information to answer this question."
        return self.generate(query, docs)


# ─── 5. Run the Evaluation ─────────────────────────────────────────────────────
rag_pipeline = TruLensRAGPipeline(rag_retriever, llm)

tru_app = TruCustomApp(
    rag_pipeline,
    app_id="Docling_Gemini_RAG_v1",
    feedbacks=[f_answer_relevance, f_context_relevance, f_groundedness]
)

with tru_app as recording:
    response = rag_pipeline.query("What is Carbon-aware Resource Scheduling?")

print("Answer Generated:\n", response)

# ─── 6. View the Results Dashboard ────────────────────────────────────────────
from trulens.dashboard import run_dashboard
run_dashboard(tru)


C:\Users\ANISH\AppData\Local\Temp\ipykernel_30556\3532126571.py:21: DeprecationWarning: Feedback is deprecated and will be removed in a future version. Use Metric instead:
  from trulens.core import Metric
  metric = Metric(implementation=fn).on_input_output()
  Feedback(google_provider.relevance_with_cot_reasons, name="Answer Relevance")
C:\Users\ANISH\AppData\Local\Temp\ipykernel_30556\3532126571.py:30: DeprecationWarning: Feedback is deprecated and will be removed in a future version. Use Metric instead:
  from trulens.core import Metric
  metric = Metric(implementation=fn).on_input_output()
  Feedback(google_provider.context_relevance_with_cot_reasons, name="Context Relevance")
C:\Users\ANISH\AppData\Local\Temp\ipykernel_30556\3532126571.py:39: DeprecationWarning: Feedback is deprecated and will be removed in a future version. Use Metric instead:
  from trulens.core import Metric
  metric = Metric(implementation=fn).on_input_output()
  Feedback(google_provider.groundedness_measure_

Retrieving documents for query: 'What is Carbon-aware Resource Scheduling?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering and re-ranking)
Answer Generated:
 Carbon-aware resource scheduling, also known as carbon-aware computing or carbon-aware scheduling, involves shifting flexible workloads toward lower-carbon electricity in time or space [main_paper.pdf]. It exploits temporal or spatial variation in grid carbon intensity, allowing delay-tolerant workloads—such as batch jobs like model training, log aggregation, backups, scientific simulations, and CI/CD workloads—to be moved toward greener periods [main_paper.pdf].

Prior work in this area has studied geographic load balancing, temporal shifting, and carbon-intensity-aware resource management [main_paper.pdf]. For example, the GREEN system is carbon-aware as it monitors per-job energy consumption and estimates carbon footprint using a carbon-intensity model [base research paper.pdf]. Its Carbon Efficiency Optimizer minimizes the cluster-wide carbon footprint by dynamically 

Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:54325 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

WARNI [trulens.feedback.computer] feedback_name=Answer Relevance, record=4938106b-116b-4e5f-b853-eb106d84e636, span_group=None had an error during computation:
Endpoint GoogleEndpoint request failed 4 time(s): 
	400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Invalid JSON payload received. Unknown name "additional_properties" at \'generation_config.response_schema\': Cannot find field.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.BadRequest', 'fieldViolations': [{'field': 'generation_config.response_schema', 'description': 'Invalid JSON payload received. Unknown name "additional_properties" at \'generation_config.response_schema\': Cannot find field.'}]}]}}
	400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Invalid JSON payload received. Unknown name "additional_properties" at \'generation_config.response_schema\': Cannot find field.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.BadReques